In [ ]:
# Assumendo che la classe PeakRegionDetector sia già definita
# (inserire qui la definizione completa della classe se necessario)

# Funzione per creare dati di esempio
def create_sample_data(shape=(1, 200), peak_type='gaussian'):
    """
    Crea dati di esempio con un picco simulato.
    
    Parameters:
    -----------
    shape : tuple
        Forma del campione (dovrebbe essere (1, 200))
    peak_type : str
        Tipo di picco da creare: 'gaussian', 'sharp', 'distributed'
    
    Returns:
    --------
    sample : np.ndarray
        Array di forma (1, 200) con rumore di fondo e picco
    """
    # Crea rumore di fondo
    np.random.seed(42)  # Per riproducibilità
    sample = np.random.normal(0.1, 0.05, shape)  # Rumore di fondo basso
    
    # Reshape per lavorare con coordinate 2D (40, 5)
    img_2d = sample.reshape(40, 5)
    
    if peak_type == 'gaussian':
        # Crea un picco gaussiano centrato
        center_r, center_c = 20, 2
        for r in range(40):
            for c in range(5):
                distance = np.sqrt((r - center_r)**2 + (c - center_c)**2)
                intensity = 2.0 * np.exp(-distance**2 / (2 * 2.5**2))
                img_2d[r, c] += intensity
                
    elif peak_type == 'sharp':
        # Crea un picco concentrato
        peak_pixels = [(18, 2), (19, 2), (20, 2), (21, 2), (22, 2),
                      (19, 1), (20, 1), (21, 1), (19, 3), (20, 3), (21, 3)]
        for r, c in peak_pixels:
            img_2d[r, c] += np.random.uniform(1.5, 3.0)
            
    elif peak_type == 'distributed':
        # Crea un picco distribuito
        peak_region = [(15, 1), (16, 1), (17, 1), (18, 1), (19, 1),
                      (15, 2), (16, 2), (17, 2), (18, 2), (19, 2),
                      (15, 3), (16, 3), (17, 3), (18, 3), (19, 3)]
        for r, c in peak_region:
            img_2d[r, c] += np.random.uniform(0.8, 1.8)
    
    return img_2d.flatten().reshape(1, 200)

# Funzione principale di esempio
def demonstrate_peak_detection():
    """Dimostra l'uso della classe PeakRegionDetector su campioni di test."""
    
    print("=== Dimostrazione Peak Region Detector ===\n")
    
    # Crea campioni di test
    samples = {
        'Gaussian Peak': create_sample_data(peak_type='gaussian'),
        'Sharp Peak': create_sample_data(peak_type='sharp'),
        'Distributed Peak': create_sample_data(peak_type='distributed')
    }
    
    # Testa diversi metodi
    methods = ['statistical', 'adaptive', 'clustering']
    
    for sample_name, sample_data in samples.items():
        print(f"\n{'='*50}")
        print(f"Analizzando: {sample_name}")
        print(f"{'='*50}")
        
        # Il campione ha forma (1, 200), prendiamo la prima riga
        ddm = sample_data[0]  # Shape (200,)
        
        for method in methods:
            print(f"\n--- Metodo: {method.upper()} ---")
            
            # Inizializza il detector
            detector = PeakRegionDetector(N=10, method=method)
            
            try:
                # Trova la regione di picco
                pixels, centroid = detector.find_peak_region(ddm)
                
                # Calcola il form factor
                img = ddm.reshape(40, 5)
                form_factor = detector._calculate_form_factor_circle(img, pixels)
                
                # Estrai features complete
                features = detector.extract_comprehensive_features(
                    pixels, centroid, form_factor, ddm
                )
                
                # Mostra risultati principali
                print(f"Numero pixel trovati: {len(pixels)}")
                print(f"Centroide: ({centroid[0]:.2f}, {centroid[1]:.2f})")
                print(f"Form factor: {form_factor:.3f}")
                
                # Mostra alcune features chiave
                key_features = [
                    'peak_intensity_mean', 'peak_intensity_std', 'peak_compactness',
                    'peak_circularity', 'peak_spatial_spread_total', 'peak_relative_intensity'
                ]
                
                print("Features principali:")
                for feat in key_features:
                    if feat in features:
                        print(f"  {feat}: {features[feat]:.3f}")
                
                # Visualizza risultati (opzionale, decommentare se si vuole la visualizzazione)
                # detector.visualize_result(ddm, pixels, centroid, 
                #                         title=f"{sample_name} - {method}")
                
            except Exception as e:
                print(f"Errore durante l'analisi: {str(e)}")
    
    return samples

# Funzione per analizzare un singolo campione
def analyze_single_sample(sample_1d, method='statistical', N=10, visualize=True):
    """
    Analizza un singolo campione di forma (200,).
    
    Parameters:
    -----------
    sample_1d : np.ndarray
        Array 1D di lunghezza 200
    method : str
        Metodo da usare ('statistical', 'adaptive', 'clustering')
    N : int
        Numero massimo di pixel nella regione
    visualize : bool
        Se True, mostra la visualizzazione dei risultati
    
    Returns:
    --------
    results : dict
        Dizionario con pixels, centroid, form_factor e features
    """
    
    print(f"Analizzando campione con metodo: {method}")
    print(f"Forma campione: {sample_1d.shape}")
    
    # Inizializza detector
    detector = PeakRegionDetector(N=N, method=method)
    
    # Trova regione di picco
    pixels, centroid = detector.find_peak_region(sample_1d)
    
    # Calcola form factor
    img = sample_1d.reshape(40, 5)
    form_factor = detector._calculate_form_factor_circle(img, pixels)
    
    # Estrai features
    features = detector.extract_comprehensive_features(
        pixels, centroid, form_factor, sample_1d
    )
    
    # Mostra visualizzazione se richiesta
    if visualize:
        detector.visualize_result(sample_1d, pixels, centroid,
                                title=f"Peak Analysis - {method}")
    
    # Prepara risultati
    results = {
        'pixels': pixels,
        'centroid': centroid,
        'form_factor': form_factor,
        'features': features,
        'method': method,
        'N': N
    }
    
    return results

# Funzione per confrontare tutti i metodi su un campione
def compare_methods(sample_1d, N=10):
    """
    Confronta tutti i metodi disponibili su un singolo campione.
    
    Parameters:
    -----------
    sample_1d : np.ndarray
        Array 1D di lunghezza 200
    N : int
        Numero massimo di pixel nella regione
    
    Returns:
    --------
    comparison : dict
        Dizionario con i risultati di tutti i metodi
    """
    
    methods = ['statistical', 'adaptive', 'clustering']
    comparison = {}
    
    print("=== Confronto Metodi ===")
    print(f"Campione shape: {sample_1d.shape}")
    print(f"Numero massimo pixel: {N}\n")
    
    for method in methods:
        print(f"Metodo: {method.upper()}")
        try:
            results = analyze_single_sample(sample_1d, method=method, N=N, visualize=False)
            comparison[method] = results
            
            print(f"  Pixel trovati: {len(results['pixels'])}")
            print(f"  Centroide: ({results['centroid'][0]:.2f}, {results['centroid'][1]:.2f})")
            print(f"  Form factor: {results['form_factor']:.3f}")
            print(f"  Intensità media: {results['features']['peak_intensity_mean']:.3f}")
            print()
            
        except Exception as e:
            print(f"  Errore: {str(e)}\n")
            comparison[method] = None
    
    return comparison

# Esempio di utilizzo
if __name__ == "__main__":
    # Dimostrazione completa
    samples = demonstrate_peak_detection()
    
    # Analisi di un campione specifico
    print("\n" + "="*60)
    print("ESEMPIO DI ANALISI SINGOLA")
    print("="*60)
    
    # Crea un campione di test
    test_sample = fit_data[2]  # Usa il primo campione di fit_data
    
    # Analizza con il metodo statistico
    results = analyze_single_sample(test_sample, method='adaptive', N=40)
    
    print(f"\nRisultati analisi:")
    print(f"Numero pixel nella regione: {len(results['pixels'])}")
    print(f"Coordinate pixel: {results['pixels']}")
    print(f"Centroide: {results['centroid']}")
    print(f"Form factor: {results['form_factor']:.4f}")
    
    # Mostra alcune features estratte
    print(f"\nAlcune features estratte:")
    for key, value in list(results['features'].items())[:10]:
        print(f"{key}: {value:.4f}")
    
    # Confronta tutti i metodi
    print("\n" + "="*60)
    print("CONFRONTO TRA METODI")
    print("="*60)
    
    comparison = compare_methods(test_sample, N=10)